In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

In [3]:
data = pd.read_csv(path_to_data / 'preprocessed_data.csv')

In [4]:
data["LOG_INCOME"] = np.log1p(data["AMT_INCOME_TOTAL"])
data = data.drop(columns=["AMT_INCOME_TOTAL"])

In [5]:
data["LOG_INCOME_PER_PERSON"] = data["LOG_INCOME"] / data["CNT_FAM_MEMBERS"]

In [6]:
data["DEPENDENCY_RATIO"] = data["CNT_CHILDREN"] / (data["CNT_FAM_MEMBERS"] + 1e-6)

In [7]:
data = data.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED"])

In [8]:
def exp_bin(x):
    if x <= 0: return 0
    if x <= 5: return 1
    if x <= 10: return 2
    if x <= 20: return 3
    return 4

data["EXPERIENCE_BIN"] = data["EXPERIENCE"].apply(exp_bin)

In [9]:
def age_bin(a):
    if a < 25: return 0
    if a < 35: return 1
    if a < 50: return 2
    if a < 65: return 3
    return 4

data["AGE_BIN"] = data["AGE"].apply(age_bin)

In [10]:
data["AGE_X_LOG_INCOME"] = data["AGE"] * data["LOG_INCOME"]
data["EXP_X_INCOME"] = data["EXPERIENCE"] * data["LOG_INCOME"]

In [11]:
def winsorize(s):
    return s.clip(s.quantile(0.01), s.quantile(0.99))

for col in ["LOG_INCOME", "LOG_INCOME_PER_PERSON", "EXPERIENCE"]:
    data[col] = winsorize(data[col])

In [12]:
data["HIGH_INCOME_FLAG"] = (data["LOG_INCOME"] > data["LOG_INCOME"].quantile(0.98)).astype(int)

In [13]:
counts = data["NAME_INCOME_TYPE"].value_counts(normalize=True)
rare = counts[counts < 0.005].index
data["NAME_INCOME_TYPE"] = data["NAME_INCOME_TYPE"].replace(
    rare, "Other"
)

In [14]:
data["INCOME_BIN"] = pd.qcut(data["LOG_INCOME"], q=4, labels=False)

In [15]:
data["ADULTS"] = data["CNT_FAM_MEMBERS"] - data["CNT_CHILDREN"]
data["ADULTS"] = data["ADULTS"].clip(lower=1)

In [16]:
data["EXP_AGE_RATIO"] = data["EXPERIENCE"] / (data["AGE"] + 1e-6)

In [17]:
data["STABILITY_SCORE"] = (
    -0.3 * data["DEPENDENCY_RATIO"] +
     0.4 * data["EXPERIENCE_BIN"] +
     0.3 * (4 - data["AGE_BIN"])
)

In [18]:
data["CHILD_BIN"] = pd.cut(
    data["CNT_CHILDREN"],
    bins=[0,1,3,10],
    labels=[0,1,2]
)

In [19]:
data['CHILD_BIN'].value_counts()

CHILD_BIN
0    7492
1    3675
2      85
Name: count, dtype: int64

In [20]:
data["CHILD_BIN"] = data["CHILD_BIN"].astype("float").astype("Int64")

In [21]:
data.shape

(36457, 30)

In [22]:
data["bad"].value_counts()

bad
0    35841
1      616
Name: count, dtype: int64

In [23]:
object_cols = data.select_dtypes(include="object").columns

for col in object_cols:
    data[col] = data[col].astype("category").cat.codes

In [24]:
from sklearn.preprocessing import LabelEncoder

for col in object_cols:
    le = LabelEncoder()
    data[col] = data[col].astype(str)   # страховка
    data[col] = le.fit_transform(data[col])

In [25]:
data

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,...,EXPERIENCE_BIN,AGE_BIN,AGE_X_LOG_INCOME,EXP_X_INCOME,HIGH_INCOME_FLAG,INCOME_BIN,ADULTS,EXP_AGE_RATIO,STABILITY_SCORE,CHILD_BIN
0,5008804,1,1,1,0,3,1,0,4,1,...,3,1,414.902781,155.588543,0,3,2,0.375000,2.1,<NA>
1,5008805,1,1,1,0,3,1,0,4,1,...,3,1,414.902781,155.588543,0,3,2,0.375000,2.1,<NA>
2,5008806,1,1,1,0,3,4,1,1,1,...,1,3,674.581609,34.892152,0,0,2,0.051724,0.7,<NA>
3,5008808,0,0,1,0,0,4,3,1,1,...,2,3,650.321409,100.049448,0,3,1,0.153846,1.1,<NA>
4,5008809,0,0,1,0,0,4,3,1,1,...,2,3,650.321409,100.049448,0,3,1,0.153846,1.1,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,1,1,1,0,3,4,1,1,1,...,2,2,595.035561,75.961987,0,3,2,0.127660,1.4,<NA>
36453,5149834,0,0,1,0,0,1,1,1,1,...,1,1,394.917174,35.901561,0,1,2,0.090909,1.3,<NA>
36454,5149838,0,0,1,0,1,1,1,1,1,...,1,1,394.917174,35.901561,0,1,2,0.090909,1.3,<NA>
36455,5150049,0,0,1,0,3,4,1,1,1,...,1,2,615.193576,12.554971,0,3,2,0.020408,1.0,<NA>


In [26]:
data.to_csv(path_to_data / "final_clean_data.csv", index=False)